In [1]:
import pandas as pd
import numpy as np
from datetime import date,datetime
from geopy.distance import geodesic
import warnings
import copy                                    

warnings.filterwarnings('ignore')

In [2]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [3]:
# database File
# df=pd.read_csv('elvisbuffalony2024obweekday_export_odbc.csv')
# KingElvis Dataframe
# ke_df=pd.read_excel("Buffalo_NY_OB_KINGElvis.xlsx",sheet_name='Elvis_Review')
# Details File Stops Sheet
# detail_df=pd.read_excel('details_buffalo_excel_template (1).xlsx',sheet_name='STOPS')

In [4]:
# database File
df=pd.read_csv('elvischarleston_sc_obweekday_export_odbc.csv')
# KingElvis Dataframe
ke_df=pd.read_excel("CARTA_OB_KINGElvis.xlsx",sheet_name='Elvis_Review')
# Details File Stops Sheet
detail_df=pd.read_excel('details_CARTA_SC_od_excel.xlsx',sheet_name='STOPS')


In [5]:
df=df[df['INTERV_INIT']!='999']
df=df[df['HAVE_5_MIN_FOR_SURVECode']==1]
ke_df=ke_df[ke_df['INTERV_INIT']!='999']
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['HAVE_5_MIN_FOR_SURVECode']==1]

In [6]:
stop_on_column_check=['stoponaddr']
stop_off_column_check=['stopoffaddr']
stop_on_id_column_check=['stoponclntid']
stop_off_id_column_check=['stopoffclntid']
stop_on_id_column=check_all_characters_present(df,stop_on_id_column_check)
stop_off_id_column=check_all_characters_present(df,stop_off_id_column_check)
stop_on_column=check_all_characters_present(df,stop_on_column_check)
stop_off_column=check_all_characters_present(df,stop_off_column_check)
stop_off_column,stop_on_column,stop_off_id_column,stop_on_id_column

(['STOP_OFF_ADDR'], ['STOP_ON_ADDR'], ['STOP_OFF_CLNTID'], ['STOP_ON_CLNTID'])

In [7]:
route_surveyed_column_check=['routesurveyedcode']
route_surveyed_column=check_all_characters_present(ke_df,route_surveyed_column_check)
route_surveyed_column

['ROUTE_SURVEYEDCode']

In [8]:
stop_on_lat_lon_columns_check=['stoponlat','stoponlong']
stop_off_lat_lon_columns_check=['stopofflat','stopofflong']
stop_on_lat_lon_columns=check_all_characters_present(df,stop_on_lat_lon_columns_check)
stop_off_lat_lon_columns=check_all_characters_present(df,stop_off_lat_lon_columns_check)
stop_off_lat_lon_columns.sort()
stop_on_lat_lon_columns.sort()
stop_off_lat_lon_columns,stop_on_lat_lon_columns

(['STOP_OFF_LAT', 'STOP_OFF_LONG'], ['STOP_ON_LAT', 'STOP_ON_LONG'])

In [9]:
columns_to_add = ['id', *route_surveyed_column,*stop_off_column, *stop_on_column, *stop_off_id_column, *stop_on_id_column,*stop_off_lat_lon_columns,*stop_on_lat_lon_columns]
columns_to_add

['id',
 'ROUTE_SURVEYEDCode',
 'STOP_OFF_ADDR',
 'STOP_ON_ADDR',
 'STOP_OFF_CLNTID',
 'STOP_ON_CLNTID',
 'STOP_OFF_LAT',
 'STOP_OFF_LONG',
 'STOP_ON_LAT',
 'STOP_ON_LONG']

In [10]:
origin_address_lat_column=['originaddresslat']
origin_address_long_column=['originaddresslong']
origin_address_lat=check_all_characters_present(df,origin_address_lat_column)
origin_address_long=check_all_characters_present(df,origin_address_long_column)
origin_address_lat,origin_address_long

(['ORIGIN_ADDRESS_LAT'], ['ORIGIN_ADDRESS_LONG'])

In [11]:
ke_df.rename(columns={'ROUTE_SURVEYEDCode': 'ROUTE_SURVEYEDCode_KE'}, inplace=True)


In [12]:
df=df[df['id']==72]

In [13]:
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_DATE,ELVIS_USER_CHANGE_7_C_FIELD,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS
44,72,2024-10-28 12:25:09,76,en,2024-10-28 12:00:16,2024-10-28 12:25:09,172.59.218.137,https://charleston-sc.etc-research.com/index.p...,-14400,ANC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# columns_to_add = ['id', *origin_address_lat,*origin_address_long]
columns_to_add

['id',
 'ROUTE_SURVEYEDCode',
 'STOP_OFF_ADDR',
 'STOP_ON_ADDR',
 'STOP_OFF_CLNTID',
 'STOP_ON_CLNTID',
 'STOP_OFF_LAT',
 'STOP_OFF_LONG',
 'STOP_ON_LAT',
 'STOP_ON_LONG']

In [15]:
ke_df=ke_df[ke_df['id']==72]

In [16]:
# Merge without prefixes or suffixes
ke_df = pd.merge(ke_df, df[columns_to_add], on='id', how='left')

In [17]:
ke_df

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,FINAL_REVIEWER2,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,...,RECORD_INFO,ROUTE_SURVEYEDCode,STOP_OFF_ADDR,STOP_ON_ADDR,STOP_OFF_CLNTID,STOP_ON_CLNTID,STOP_OFF_LAT,STOP_OFF_LONG,STOP_ON_LAT,STOP_ON_LONG
0,2024-11-08,72,HereAPI,Priyanka,NaN,Use,NaN,NaN,Elvis_Review,Elvis_Review,...,NaN,CAR_1_213_00,America St / Columbus St,Visitors Center on John St,CAR_1_213_00_798915,CAR_1_213_00_764307,32.796148,-79.936105,32.78894,-79.936319


In [18]:
ke_df[[route_surveyed_column[0]]].head(2)

,ROUTE_SURVEYEDCode
0,CAR_1_213_00


In [19]:
ke_df['ROUTE_SURVEYEDCode_SPLITED']=ke_df['ROUTE_SURVEYEDCode'].apply(lambda x : '_'.join(str(x).split('_')[0:-1]))
ke_df[['ROUTE_SURVEYEDCode_SPLITED']]

,ROUTE_SURVEYEDCode_SPLITED
0,CAR_1_213


In [20]:
detail_df['ETC_ROUTE_ID_SPLITED']=detail_df['ETC_ROUTE_ID'].apply(lambda x : '_'.join(str(x).split('_')[0:-1]))
detail_df[['ETC_ROUTE_ID_SPLITED']].head(2)

,ETC_ROUTE_ID_SPLITED
0,CAR_1_10
1,CAR_1_10


In [21]:
detail_df['ETC_ROUTE_DIRECTION']=detail_df['ETC_ROUTE_ID'].apply(lambda x : str(x).split('_')[-1])
detail_df[['ETC_ROUTE_DIRECTION']].head(2)

,ETC_ROUTE_DIRECTION
0,00
1,00


In [22]:

detail_df['ETC_STOP_DIRECTION']=detail_df['ETC_STOP_ID'].apply(lambda x : str(x).split('_')[-2])
detail_df[['ETC_STOP_DIRECTION']].head(2)

,ETC_STOP_DIRECTION
0,00
1,00


In [23]:
detail_df[detail_df['ETC_ROUTE_ID_SPLITED']=='NFT_1_19']

,agency,gtfs_ver,gtfs_date,route_short_name,route_long_name,direction_id,route_dir,seq_fixed,stop_id,stop_name,...,stop_lat6,stop_lon6,ETC_STOP_ID,ETC_STOP_NAME,notes,ETC_ROUTE_ID,ETC_ROUTE_NAME,ETC_ROUTE_ID_SPLITED,ETC_ROUTE_DIRECTION,ETC_STOP_DIRECTION


In [24]:

def get_distance_between_coordinates(lat1, lon1, lat2, lon2):
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
        
        coords_1 = (lat1, lon1)
        coords_2 = (lat2, lon2)
        
        distance = geodesic(coords_1, coords_2).miles
        return distance
    except (ValueError, TypeError) as e:
        # Handle the exception here
        print(f"Error calculating distance: {e}")  # Change to the desired distance unit

# The Below Block is used to Find Nearest STOP_ON Stop Using ORIGIN

In [25]:
# # Iterate through df rows to get the STOP_ON points
# for _, row in ke_df.iterrows():
#     nearest_stop_seq = []    
    
#     origin_lat = row[origin_address_lat[0]]
#     origin_long = row[origin_address_long[0]]
#     print(origin_lat,origin_long)
# #     if pd.isna(origin_lat) or pd.isna(origin_long):
# #         continue 
    
#     # Filtered data
# #     filtered_df = detail_df[detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED']][
# #         ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
# #     ]
#     filtered_df = detail_df[detail_df['ETC_ROUTE_ID'] == row['ROUTE_SURVEYEDCode']][
#         ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
#     ]
    
#     # List to store distances
#     distances = []
    
#     # Calculate distances for all rows in filtered_df
#     for _, detail_row in filtered_df.iterrows():
#         stop_lat6 = detail_row['stop_lat6']
#         stop_lon6 = detail_row['stop_lon6']
        
#         # Compute distance
#         distance = get_distance_between_coordinates(origin_lat, origin_long, stop_lat6, stop_lon6)
        
#         # Skip distance if it is 0
#         if distance == 0:
#             continue
        
#         distances.append((distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'],detail_row['ETC_STOP_NAME'],detail_row['stop_lat6'],detail_row['stop_lon6']))
    
#     # Find the nearest stop (minimum distance)
#     if distances:
#         nearest_stop = min(distances, key=lambda x: x[0])  # x[0] is the distance
#         nearest_stop_seq.append(nearest_stop)
    
#     # Process nearest_stop_seq as needed
#     print(f"Nearest stop details for row: {nearest_stop_seq}")
#     if nearest_stop_seq:
#         ke_df.loc[row.name, 'STOP_ON_ADDRESS'] = nearest_stop_seq[0][3]  # ETC_STOP_NAME
#         ke_df.loc[row.name, 'STOP_ON_SEQ'] = nearest_stop_seq[0][1]      # seq_fixed
#         ke_df.loc[row.name, 'STOP_ON_CLINTID'] = nearest_stop_seq[0][2]  # ETC_STOP_ID
#         ke_df.loc[row.name, 'STOP_ON_LAT'] = nearest_stop_seq[0][4]      # stop_lat6
#         ke_df.loc[row.name, 'STOP_ON_LONG'] = nearest_stop_seq[0][5]     # stop_lon6

# The Below Block is used to nearest STOP_ON STOP using existing STOP_ON Value

In [26]:
# Iterate through df rows to get the STOP_ON points
for _, row in ke_df.iterrows():
    nearest_stop_seq = []    
    
    stop_on_id=row[stop_on_id_column[0]]    
    
    stop_on_lat = row[stop_on_lat_lon_columns[0]]
    stop_on_long = row[stop_on_lat_lon_columns[1]]
#     if pd.isna(origin_lat) or pd.isna(origin_long):
#         continue 
    
    # Filtered data
#     filtered_df = detail_df[detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED']][
#         ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
#     ]
    filtered_df = detail_df[detail_df['ETC_ROUTE_ID'] == row['ROUTE_SURVEYEDCode']][
        ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
    ]
    
    # List to store distances
    distances = []
    
    # Calculate distances for all rows in filtered_df
    for _, detail_row in filtered_df.iterrows():
        stop_lat6 = detail_row['stop_lat6']
        stop_lon6 = detail_row['stop_lon6']
        
        # Compute distance
        distance = get_distance_between_coordinates(stop_on_lat, stop_on_long, stop_lat6, stop_lon6)
        
        # Skip distance if it is 0
#         if distance == 0:
#             continue
        
        distances.append((distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'],detail_row['ETC_STOP_NAME'],detail_row['stop_lat6'],detail_row['stop_lon6']))
    
    # Find the nearest stop (minimum distance)
    if distances:
        nearest_stop = min(distances, key=lambda x: x[0])  # x[0] is the distance
        nearest_stop_seq.append(nearest_stop)
    
    # Process nearest_stop_seq as needed
#     print(f"Nearest stop details for row: {nearest_stop_seq}")
    if nearest_stop_seq:
        ke_df.loc[row.name, 'STOP_ON_ADDR_NEW'] = nearest_stop_seq[0][3]  # ETC_STOP_NAME
        ke_df.loc[row.name, 'STOP_ON_SEQ'] = nearest_stop_seq[0][1]      # seq_fixed
        ke_df.loc[row.name, 'STOP_ON_CLINTID_NEW'] = nearest_stop_seq[0][2]  # ETC_STOP_ID
        ke_df.loc[row.name, 'STOP_ON_LAT_NEW'] = nearest_stop_seq[0][4]      # stop_lat6
        ke_df.loc[row.name, 'STOP_ON_LONG_NEW'] = nearest_stop_seq[0][5]     # stop_lon6

In [27]:
ke_df

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,FINAL_REVIEWER2,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,...,STOP_OFF_LAT,STOP_OFF_LONG,STOP_ON_LAT,STOP_ON_LONG,ROUTE_SURVEYEDCode_SPLITED,STOP_ON_ADDR_NEW,STOP_ON_SEQ,STOP_ON_CLINTID_NEW,STOP_ON_LAT_NEW,STOP_ON_LONG_NEW
0,2024-11-08,72,HereAPI,Priyanka,NaN,Use,NaN,NaN,Elvis_Review,Elvis_Review,...,32.796148,-79.936105,32.78894,-79.936319,CAR_1_213,Visitors Center on John St,7.0,CAR_1_213_00_764307,32.78894,-79.936319


In [28]:
ke_df[['STOP_ON_CLNTID','STOP_ON_CLINTID_NEW']]

,STOP_ON_CLNTID,STOP_ON_CLINTID_NEW
0,CAR_1_213_00_764307,CAR_1_213_00_764307


In [29]:
ke_df[['STOP_ON_LAT','STOP_ON_LONG']]

,STOP_ON_LAT,STOP_ON_LONG
0,32.78894,-79.936319


In [30]:
# Iterate through df rows to get the STOP_OFF points
# Iterate through new_df rows
for _, row in ke_df.iterrows():
    nearest_stop_seq = []
    
#     stop_on_id=row[stop_on_id_column[0]]    
    stop_off_lat = row[stop_off_lat_lon_columns[0]]
    stop_off_long = row[stop_off_lat_lon_columns[1]]
#     print(row['id'],stop_off_lat,stop_off_long)
#     stop_on_lat = row['STOP_ON_LAT_NEW']
#     stop_on_long = row['STOP_ON_LONG_NEW']
    if pd.isna(stop_off_lat) or pd.isna(stop_off_long):
        continue
    stop_on_direction = str(row['STOP_ON_CLINTID_NEW']).split('_')[-2] if len(str(row['STOP_ON_CLINTID_NEW']).split('_')) >= 2 else None
    if stop_on_direction is None:
        # Skip the current iteration if the direction cannot be determined
        continue
#     print(f'{stop_on_direction=}')
    # Filtered data
#     filtered_df = detail_df[(detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED'])&(detail_df['ETC_STOP_DIRECTION']==stop_on_direction)][
#         ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
#     ]
    filtered_df = detail_df[(detail_df['ETC_ROUTE_ID'] == row['ROUTE_SURVEYEDCode'])&(detail_df['ETC_STOP_DIRECTION']==stop_on_direction)][
        ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
    ]
#     filtered_df = detail_df[detail_df['ETC_ROUTE_ID'] == row['ROUTE_SURVEYEDCode']][
#         ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']

    # List to store distances
    distances = []
    
    # Calculate distances for all rows in filtered_df
    for _, detail_row in filtered_df.iterrows():
        stop_lat6 = detail_row['stop_lat6']
        stop_lon6 = detail_row['stop_lon6']
        
        # Compute distance
        distance = get_distance_between_coordinates(stop_off_lat, stop_off_long, stop_lat6, stop_lon6)
        # Skip distance if it is 0
#         if distance == 0:
#             continue
#         if distance>0.5:
        distances.append((distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'],detail_row['ETC_STOP_NAME'],detail_row['stop_lat6'],detail_row['stop_lon6']))
    
    # Find the nearest stop (minimum distance)
    if distances:
        nearest_stop = min(distances, key=lambda x: x[0])  # x[0] is the distance
        nearest_stop_seq.append(nearest_stop)
#     print(len(distances))
    # Process nearest_stop_seq as needed
#     print(f"Nearest stop details for row: {nearest_stop_seq}")
    if nearest_stop_seq:
        ke_df.loc[row.name, 'STOP_OFF_ADDRESS_NEW'] = nearest_stop_seq[0][3]  # ETC_STOP_NAME
        ke_df.loc[row.name, 'STOP_OFF_SEQ'] = nearest_stop_seq[0][1]      # seq_fixed
        ke_df.loc[row.name, 'STOP_OFF_CLINTID_NEW'] = nearest_stop_seq[0][2]  # ETC_STOP_ID
        ke_df.loc[row.name, 'STOP_OFF_LAT_NEW'] = nearest_stop_seq[0][4]      # stop_lat6
        ke_df.loc[row.name, 'STOP_OFF_LONG_NEW'] = nearest_stop_seq[0][5]     # stop_lon6

In [31]:
ke_df[[route_surveyed_column[0],'STOP_OFF_CLNTID','STOP_OFF_CLINTID_NEW','STOP_ON_CLNTID','STOP_ON_CLINTID_NEW']]

,ROUTE_SURVEYEDCode,STOP_OFF_CLNTID,STOP_OFF_CLINTID_NEW,STOP_ON_CLNTID,STOP_ON_CLINTID_NEW
0,CAR_1_213_00,CAR_1_213_00_798915,CAR_1_213_00_798915,CAR_1_213_00_764307,CAR_1_213_00_764307


In [32]:
ke_df['SEQ_DIFFERENCE']=ke_df['STOP_OFF_SEQ']-ke_df['STOP_ON_SEQ']
ke_df[['STOP_OFF_CLINTID_NEW','STOP_ON_CLINTID_NEW','SEQ_DIFFERENCE']]

,STOP_OFF_CLINTID_NEW,STOP_ON_CLINTID_NEW,SEQ_DIFFERENCE
0,CAR_1_213_00_798915,CAR_1_213_00_764307,-4.0


In [33]:
ke_df

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,FINAL_REVIEWER2,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,...,STOP_ON_SEQ,STOP_ON_CLINTID_NEW,STOP_ON_LAT_NEW,STOP_ON_LONG_NEW,STOP_OFF_ADDRESS_NEW,STOP_OFF_SEQ,STOP_OFF_CLINTID_NEW,STOP_OFF_LAT_NEW,STOP_OFF_LONG_NEW,SEQ_DIFFERENCE
0,2024-11-08,72,HereAPI,Priyanka,NaN,Use,NaN,NaN,Elvis_Review,Elvis_Review,...,7.0,CAR_1_213_00_764307,32.78894,-79.936319,America St / Columbus St,3.0,CAR_1_213_00_798915,32.796148,-79.936105,-4.0


In [34]:

ke_df[['ROUTE_SURVEYEDCode']]

,ROUTE_SURVEYEDCode
0,CAR_1_213_00


In [35]:
'NFT_1_19_01_31350'.split('_')[-2]

'01'

In [36]:
test_df=detail_df[(detail_df['ETC_ROUTE_ID_SPLITED'] == 'NFT_1_19') &
            (detail_df['ETC_STOP_DIRECTION'] != '01')
        ][['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID', 'ETC_STOP_NAME']]

In [37]:
nearest_stop

(0.0,
 3.0,
 'CAR_1_213_00_798915',
 'America St / Columbus St',
 32.796148,
 -79.936105)

In [38]:
for _,row in ke_df.iterrows():
    nearest_stop_on_seq=[]
    nearest_stop_off_seq=[]
    route_code = row[route_surveyed_column[0]]
    if row['SEQ_DIFFERENCE'] < 0:
        
        stop_on_lat = row['STOP_ON_LAT_NEW']
        stop_on_long = row['STOP_ON_LONG_NEW']

        stop_off_lat = row['STOP_OFF_LAT_NEW']
        stop_off_long = row['STOP_OFF_LONG_NEW']
        
#         print(row['STOP_ON_CLINTID_NEW'])
#         print(row['STOP_OFF_CLINTID_NEW'])
        stop_on_direction = row[ 'STOP_ON_CLINTID_NEW'].split('_')[-2]
        stop_off_direction = row[ 'STOP_OFF_CLINTID_NEW'].split('_')[-2]
#         print(stop_on_direction)
#         print(stop_off_direction)
#         print(route_code)
        new_route_code = (
            f"{'_'.join(route_code.split('_')[:-1])}_01" 
            if route_code.split('_')[-1] == '00' 
            else f"{'_'.join(route_code.split('_')[:-1])}_00"
        )
        ke_df.loc[row.name, 'ROUTE_SURVEYEDCode_New'] = route_code
        ke_df.loc[row.name, 'ROUTE_SURVEYED_NEW'] = ke_df.loc[row.name, 'ROUTE_SURVEYED']
        new_route_name_row = detail_df[detail_df['ETC_ROUTE_ID'] == new_route_code]
        if not new_route_name_row.empty:
            new_route_name = new_route_name_row['ETC_ROUTE_NAME'].iloc[0]
            
            ke_df.loc[row.name, 'ROUTE_SURVEYEDCode_New'] = new_route_code
            ke_df.loc[row.name, 'ROUTE_SURVEYED_NEW'] = new_route_name
#             print(new_route_code)
            filtered_stop_on_df = detail_df[(detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED'])&(detail_df['ETC_STOP_DIRECTION']!=stop_on_direction)][
            ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
        ]
            filtered_stop_off_df = detail_df[(detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED'])&(detail_df['ETC_STOP_DIRECTION']!=stop_off_direction)][
            ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
        ]
    #         print(filtered_stop_on_df)
            stop_on_distances = []
#             print(stop_on_distances)
            # Calculate distances for all rows in filtered_df
            for _, detail_row in filtered_stop_on_df.iterrows():
                stop_lat6 = detail_row['stop_lat6']
                stop_lon6 = detail_row['stop_lon6']
    #             print(detail_row['ETC_STOP_ID'])
                # Compute distance
                stop_on_distance = get_distance_between_coordinates(stop_on_lat, stop_on_long,stop_lat6, stop_lon6)

                # Skip distance if it is 0
    #             if stop_on_distance == 0:
    #                 continue

                stop_on_distances.append((stop_on_distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'],detail_row['ETC_STOP_NAME'],detail_row['stop_lat6'],detail_row['stop_lon6']))
            # Find the nearest stop (minimum distance)
            if stop_on_distances:
                nearest_stop_on = min(stop_on_distances, key=lambda x: x[0])  # x[0] is the distance
                nearest_stop_on_seq.append(nearest_stop_on)
            print(f"Nearest stop details for row: {nearest_stop_on_seq=}")
            if nearest_stop_on_seq:
                ke_df.loc[row.name, 'STOP_ON_ADDRESS_NEW'] = nearest_stop_on_seq[0][3]  # ETC_STOP_NAME
                ke_df.loc[row.name, 'STOP_ON_SEQ'] = nearest_stop_on_seq[0][1]      # seq_fixed
                ke_df.loc[row.name, 'STOP_ON_CLINTID_NEW'] = nearest_stop_on_seq[0][2]  # ETC_STOP_ID
                ke_df.loc[row.name, 'STOP_ON_LAT_NEW'] = nearest_stop_on_seq[0][4]      # stop_lat6
                ke_df.loc[row.name, 'STOP_ON_LONG_NEW'] = nearest_stop_on_seq[0][5]     # stop_lon6
            stop_off_distances = []
#             print(f'{stop_off_distances=}')
            # Calculate distances for all rows in filtered_df
            for _, detail_row in filtered_stop_off_df.iterrows():
                stop_lat6 = detail_row['stop_lat6']
                stop_lon6 = detail_row['stop_lon6']
    #             print(detail_row['ETC_STOP_ID'])
                # Compute distance
                stop_off_distance = get_distance_between_coordinates(stop_off_lat, stop_off_long,stop_lat6, stop_lon6)

    #             Skip distance if it is 0
    #             if stop_off_distance == 0:
    #                 continue

                stop_off_distances.append((stop_off_distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'],detail_row['ETC_STOP_NAME'],detail_row['stop_lat6'],detail_row['stop_lon6']))
            # Find the nearest stop (minimum distance)
            if stop_off_distances:
                nearest_stop_off = min(stop_off_distances, key=lambda x: x[0])  # x[0] is the distance
                nearest_stop_off_seq.append(nearest_stop_off)
            print(f"Nearest stop details for row: {nearest_stop_off_seq=}")
            if nearest_stop_off_seq:
                ke_df.loc[row.name, 'STOP_OFF_ADDRESS_NEW'] = nearest_stop_off_seq[0][3]  # ETC_STOP_NAME
                ke_df.loc[row.name, 'STOP_OFF_SEQ'] = nearest_stop_off_seq[0][1]      # seq_fixed
                ke_df.loc[row.name, 'STOP_OFF_CLINTID_NEW'] = nearest_stop_off_seq[0][2]  # ETC_STOP_ID
                ke_df.loc[row.name, 'STOP_OFF_LAT_NEW'] = nearest_stop_off_seq[0][4]      # stop_lat6
                ke_df.loc[row.name, 'STOP_OFF_LONG_NEW'] = nearest_stop_off_seq[0][5]
    else:
        ke_df.loc[row.name, 'ROUTE_SURVEYEDCode_New'] = route_code
        ke_df.loc[row.name, 'ROUTE_SURVEYED_NEW'] = ke_df.loc[row.name, 'ROUTE_SURVEYED']


In [50]:
for _,row in ke_df.iterrows():
    nearest_stop_on_seq=[]
    nearest_stop_off_seq=[]
    route_code = row[route_surveyed_column[0]]
    if row['SEQ_DIFFERENCE'] < 0:
        
        stop_on_lat = row['STOP_ON_LAT_NEW']
        stop_on_long = row['STOP_ON_LONG_NEW']

        stop_off_lat = row['STOP_OFF_LAT_NEW']
        stop_off_long = row['STOP_OFF_LONG_NEW']
        
#         print(row['STOP_ON_CLINTID_NEW'])
#         print(row['STOP_OFF_CLINTID_NEW'])
        stop_on_direction = row[ 'STOP_ON_CLINTID_NEW'].split('_')[-2]
        stop_off_direction = row[ 'STOP_OFF_CLINTID_NEW'].split('_')[-2]
#         print(stop_on_direction)
#         print(stop_off_direction)
#         print(route_code)
        new_route_code = (
            f"{'_'.join(route_code.split('_')[:-1])}_01" 
            if route_code.split('_')[-1] == '00' 
            else f"{'_'.join(route_code.split('_')[:-1])}_00"
        )
        ke_df.loc[row.name, 'ROUTE_SURVEYEDCode_New'] = route_code
        ke_df.loc[row.name, 'ROUTE_SURVEYED_NEW'] = ke_df.loc[row.name, 'ROUTE_SURVEYED']
        new_route_name_row = detail_df[detail_df['ETC_ROUTE_ID'] == new_route_code]
        if not new_route_name_row.empty:
            new_route_name = new_route_name_row['ETC_ROUTE_NAME'].iloc[0]
            
            ke_df.loc[row.name, 'ROUTE_SURVEYEDCode_New'] = new_route_code
            ke_df.loc[row.name, 'ROUTE_SURVEYED_NEW'] = new_route_name
#             print(new_route_code)
            filtered_stop_on_df = detail_df[(detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED'])&(detail_df['ETC_STOP_DIRECTION']!=stop_on_direction)][
            ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
        ]
            filtered_stop_off_df = detail_df[(detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED'])&(detail_df['ETC_STOP_DIRECTION']!=stop_off_direction)][
            ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
        ]
    #         print(filtered_stop_on_df)
            stop_on_distances = []
#             print(stop_on_distances)
            # Calculate distances for all rows in filtered_df
            for _, detail_row in filtered_stop_on_df.iterrows():
                stop_lat6 = detail_row['stop_lat6']
                stop_lon6 = detail_row['stop_lon6']
    #             print(detail_row['ETC_STOP_ID'])
                # Compute distance
                stop_on_distance = get_distance_between_coordinates(stop_on_lat, stop_on_long,stop_lat6, stop_lon6)

                # Skip distance if it is 0
    #             if stop_on_distance == 0:
    #                 continue

                stop_on_distances.append((stop_on_distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'],detail_row['ETC_STOP_NAME'],detail_row['stop_lat6'],detail_row['stop_lon6']))
            # Find the nearest stop (minimum distance)
            if stop_on_distances:
                nearest_stop_on = min(stop_on_distances, key=lambda x: x[0])  # x[0] is the distance
                nearest_stop_on_seq.append(nearest_stop_on)
            print(f"Nearest stop details for row: {nearest_stop_on_seq=}")
            if nearest_stop_on_seq:
                ke_df.loc[row.name, 'STOP_ON_ADDRESS_NEW'] = nearest_stop_on_seq[0][3]  # ETC_STOP_NAME
                ke_df.loc[row.name, 'STOP_ON_SEQ'] = nearest_stop_on_seq[0][1]      # seq_fixed
                ke_df.loc[row.name, 'STOP_ON_CLINTID_NEW'] = nearest_stop_on_seq[0][2]  # ETC_STOP_ID
                ke_df.loc[row.name, 'STOP_ON_LAT_NEW'] = nearest_stop_on_seq[0][4]      # stop_lat6
                ke_df.loc[row.name, 'STOP_ON_LONG_NEW'] = nearest_stop_on_seq[0][5]     # stop_lon6
            stop_off_distances = []
#             print(f'{stop_off_distances=}')
            # Calculate distances for all rows in filtered_df
            for _, detail_row in filtered_stop_off_df.iterrows():
                stop_lat6 = detail_row['stop_lat6']
                stop_lon6 = detail_row['stop_lon6']
    #             print(detail_row['ETC_STOP_ID'])
                # Compute distance
                stop_off_distance = get_distance_between_coordinates(stop_off_lat, stop_off_long,stop_lat6, stop_lon6)

    #             Skip distance if it is 0
    #             if stop_off_distance == 0:
    #                 continue

                stop_off_distances.append((stop_off_distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'],detail_row['ETC_STOP_NAME'],detail_row['stop_lat6'],detail_row['stop_lon6']))
            # Find the nearest stop (minimum distance)
            if stop_off_distances:
                nearest_stop_off = min(stop_off_distances, key=lambda x: x[0])  # x[0] is the distance
                nearest_stop_off_seq.append(nearest_stop_off)
            print(f"Nearest stop details for row: {nearest_stop_off_seq=}")
            if nearest_stop_off_seq:
                ke_df.loc[row.name, 'STOP_OFF_ADDRESS_NEW'] = nearest_stop_off_seq[0][3]  # ETC_STOP_NAME
                ke_df.loc[row.name, 'STOP_OFF_SEQ'] = nearest_stop_off_seq[0][1]      # seq_fixed
                ke_df.loc[row.name, 'STOP_OFF_CLINTID_NEW'] = nearest_stop_off_seq[0][2]  # ETC_STOP_ID
                ke_df.loc[row.name, 'STOP_OFF_LAT_NEW'] = nearest_stop_off_seq[0][4]      # stop_lat6
                ke_df.loc[row.name, 'STOP_OFF_LONG_NEW'] = nearest_stop_off_seq[0][5]
        else:
            filtered_stop_on_df = detail_df[(detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED']) & (detail_df['ETC_STOP_DIRECTION'] == stop_on_direction)][
                ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID', 'ETC_STOP_NAME']
            ]

            stop_on_distances = []

            # Calculate distances for all rows in filtered_stop_on_df
            for _, detail_row in filtered_stop_on_df.iterrows():
                stop_lat6 = detail_row['stop_lat6']
                stop_lon6 = detail_row['stop_lon6']

                # Compute distance
                stop_on_distance = get_distance_between_coordinates(stop_on_lat, stop_on_long, stop_lat6, stop_lon6)

                # Include only distances greater than 0.5
                if stop_on_distance > 0.5:
                    stop_on_distances.append((stop_on_distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'], detail_row['ETC_STOP_NAME'], detail_row['stop_lat6'], detail_row['stop_lon6']))

            # Find the nearest stop (minimum distance)
            print(f'This is inside else {stop_on_distances=}')
            if stop_on_distances:
                nearest_stop_on = min(stop_on_distances, key=lambda x: x[0])  # x[0] is the distance
                nearest_stop_on_seq.append(nearest_stop_on)

            if nearest_stop_on_seq:
                ke_df.loc[row.name, 'STOP_ON_ADDRESS_NEW'] = nearest_stop_on_seq[0][3]  # ETC_STOP_NAME
                ke_df.loc[row.name, 'STOP_ON_SEQ'] = nearest_stop_on_seq[0][1]  # seq_fixed
                ke_df.loc[row.name, 'STOP_ON_CLINTID_NEW'] = nearest_stop_on_seq[0][2]  # ETC_STOP_ID
                ke_df.loc[row.name, 'STOP_ON_LAT_NEW'] = nearest_stop_on_seq[0][4]  # stop_lat6
                ke_df.loc[row.name, 'STOP_ON_LONG_NEW'] = nearest_stop_on_seq[0][5]  # stop_lon6
    else:
        ke_df.loc[row.name, 'ROUTE_SURVEYEDCode_New'] = route_code
        ke_df.loc[row.name, 'ROUTE_SURVEYED_NEW'] = ke_df.loc[row.name, 'ROUTE_SURVEYED']


This is inside else stop_on_distances=[(0.6526936182616495, 11.0, 'CAR_1_213_00_819712', 'Calhoun St / Ashley Ave', 32.783371, -79.945389), (0.8173946967891161, 12.0, 'CAR_1_213_00_800540', 'Calhoun St / Jonathan Lucas St (far side)', 32.782593, -79.948182), (0.9249426969775767, 13.0, 'CAR_1_213_00_800541', 'Courtenay Dr / Doughty St', 32.783939, -79.951065), (1.0668298063091284, 14.0, 'CAR_1_213_00_764272', 'Bee St / VA Hospital', 32.785163, -79.954093), (1.2392109347256666, 15.0, 'CAR_1_213_00_764282', 'Lockwood Dr / Westedge St', 32.786406, -79.957396), (1.3709520876093162, 16.0, 'CAR_1_213_00_764309', 'Lockwood Dr / Brittlebank Park', 32.788333, -79.959861), (1.3113941044888437, 17.0, 'CAR_1_213_00_764328', 'Fishburne St / Westedge St', 32.790471, -79.958776), (1.0131458390041306, 18.0, 'CAR_1_213_00_764345', 'Fishburne St / President St', 32.792799, -79.953115), (0.9480430663363181, 19.0, 'CAR_1_213_00_764325', 'President St / Line St', 32.79133, -79.952359), (0.8116450485171907, 

In [51]:
ke_df[['STOP_ON_CLNTID','STOP_OFF_CLNTID','ROUTE_SURVEYEDCode','ROUTE_SURVEYEDCode_New','STOP_ON_CLINTID_NEW','STOP_OFF_CLINTID_NEW']]

,STOP_ON_CLNTID,STOP_OFF_CLNTID,ROUTE_SURVEYEDCode,ROUTE_SURVEYEDCode_New,STOP_ON_CLINTID_NEW,STOP_OFF_CLINTID_NEW
0,CAR_1_213_00_764307,CAR_1_213_00_798915,CAR_1_213_00,CAR_1_213_00,CAR_1_213_00_764312,CAR_1_213_00_798915


In [53]:
ke_df.STOP_ON_CLNTID.dtype

dtype('O')

In [41]:
# ke_df.to_csv('ROUTE_DIRECTION_REFACTOR_ROUTE_SPECIFIC03.csv',index=False)

In [42]:
# for _,row in ke_df.iterrows():

#     if row['SEQ_DIFFERENCE'] < 0:
#         route_code = row[route_surveyed_column[0]]  # ROUTE_SURVEYEDCode
#         stop_on_lat = row['STOP_ON_LAT']
#         stop_on_long = row['STOP_ON_LONG']
#         stop_off_lat = row['STOP_OFF_LAT']
#         stop_off_long = row['STOP_OFF_LONG']
#         print(row['STOP_ON_CLINTID'])
#         stop_on_direction = row[ 'STOP_ON_CLINTID'].split('_')[-2]
#         print(stop_on_direction)
#         stop_off_direction = row['STOP_OFF_CLINTID'].split('_')[-2]
#         print(route_code)
#         new_route_code = (
#             f"{'_'.join(route_code.split('_')[:-1])}_01" 
#             if route_code.split('_')[-1] == '00' 
#             else f"{'_'.join(route_code.split('_')[:-1])}_00"
#         )
#         print(new_route_code)
#         # Filter stops in reverse direction for STOP_ON
#         filtered_stop_on_df = detail_df[
#             (detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED'])&(detail_df['ETC_STOP_DIRECTION'] != stop_on_direction)
#         ][['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID', 'ETC_STOP_NAME','ETC_ROUTE_ID']]
#         print(filtered_stop_on_df[['ETC_STOP_ID']].head())
#         # List to store distances
# #         print(filtered_df)
#         distances = []

#         # Calculate distances for all rows in filtered_df
#         for _, detail_row in filtered_stop_on_df.iterrows():
#             stop_lat6 = detail_row['stop_lat6']
#             stop_lon6 = detail_row['stop_lon6']
            
#             # Compute distance
#             distance = get_distance_between_coordinates(stop_on_lat, stop_on_long,stop_lat6, stop_lon6)

#             # Skip distance if it is 0
#             if distance == 0:
#                 continue

#             distances.append((distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'],detail_row['ETC_STOP_NAME'],detail_row['stop_lat6'],detail_row['stop_lon6']))
#         # Find the nearest stop (minimum distance)
#         if distances:
#             nearest_stop = min(distances, key=lambda x: x[0])  # x[0] is the distance
#             nearest_stop_seq.append(nearest_stop)
#         # Process nearest_stop_seq as needed
#         print(f"Nearest stop details for row: {nearest_stop_seq}")


In [43]:
        # Update STOP_ON details if reverse direction exists
#         if nearest_stop_on:
#             ke_df.loc[idx, 'STOP_ON_ADDRESS'] = nearest_stop_on[3]  # ETC_STOP_NAME
#             ke_df.loc[idx, 'STOP_ON_SEQ'] = nearest_stop_on[1]      # seq_fixed
#             ke_df.loc[idx, 'STOP_ON_CLINTID'] = nearest_stop_on[2]  # ETC_STOP_ID
#             ke_df.loc[idx, 'STOP_ON_LAT'] = nearest_stop_on[4]      # stop_lat6
#             ke_df.loc[idx, 'STOP_ON_LONG'] = nearest_stop_on[5]     # stop_lon6




In [44]:
get_distance_between_coordinates(42.95502,-78.820225,42.957602,-78.818517)

0.19815888743079765

In [45]:
get_distance_between_coordinates(42.955496, -78.898445,42.922333,-78.898397)

2.289212889542756

In [46]:
# STOP_ON_CLINTID_NEW=NFT_1_19_00_31350
# STOP_OFF_CLINTID_NEW=NFT_1_19_00_29730
get_distance_between_coordinates(42.9550199999998, -78.8202249999998,42.957602, -78.818517)

0.1981588874387039

In [47]:
# STOP_ON_CLINTID_NEW=NFT_1_19_00_31350
# STOP_OFF_CLINTID_NEW=NFT_1_19_00_2730
get_distance_between_coordinates(42.9550199999998, -78.8202249999998,42.926384, -78.813775)

2.003601292417177

In [48]:
# STOP_ON_CLINTID_NEW=NFT_1_8_00_42010
# STOP_OFF_CLINTID_NEW=NFT_1_8_00_42020
get_distance_between_coordinates(42.885851, -78.875607,42.884658, -78.876047)

0.0853260079500457

In [49]:
# STOP_ON_CLINTID_NEW=NFT_1_8_01_14810
# STOP_OFF_CLINTID_NEW=NFT_1_8_01_30140
get_distance_between_coordinates(42.9550199999998, -78.8202249999998,42.926384, -78.813775)

2.003601292417177